# Day 3: Fine-Tuning Concepts and LoRA/QLoRA Workflow

day 2 covered running an LLM locally, today's about actually preparing to fine-tune one. the curriculum scopes today's deliverable specifically as a dataset plus a written workflow, not a full training run, so this notebook builds a real instruction dataset from our own news headlines, splits it properly, and documents the LoRA/QLoRA workflow that would use it.

In [1]:
import pandas as pd
import json

## news_dataset.csv is the grown Week 3 dataset (446 rows), same one used since Day 5 of that week
df = pd.read_csv("news_dataset.csv")
print("Rows loaded:", len(df))
print(df["Category"].value_counts())

Rows loaded: 843
Category
Business      363
Markets       205
Technology    119
Politics       56
Energy         56
Health         44
Name: count, dtype: int64


before building the dataset itself, grew this from 278 to 446 rows by re-running Week 3's scraper. it actually broke on the way, marketscreener.com had added a promo ad element with two `<b>` tags inside one link, and the scraper's selector assumed there'd always be exactly one, so Playwright's strict mode refused to guess and threw an error. fixed by taking `.first` on that locator instead of assuming a single match. real example of a scraper being fragile to a site's layout changing, not a bug in the scraping logic itself.

In [2]:
##----- building the instruction data set -----#
##Format is Alpaca Style: INSTRUCTION -> INPUT -> OUTPUT (most common method)
##formats real fine tuning jobs, each field doing a specific job
##instuction = taks at hand, input = the specific thing to apply to it, output = the correct answer
instruction_text = "Classify this news headline into one category: Technology, Markets, Business, Politics, Health, or Energy."

examples = []
for _, row in df.iterrows():
    examples.append({
        "instruction": instruction_text,
        "input": row["Title"],
        "output": row["Category"],
    })
print(f"\nBuilt {len(examples)} instruction examples")
print("First example:", examples[0])

##JSONL = one JSON object per line, not one big array (this will be the format)
##most fine-tuning frameworks including LoRA/QLoRA expect this directly
with open("finetune_dataset.jsonl", "w") as f:
    for ex in examples:
        f.write(json.dumps(ex) + "\n")
print("Saved finetune_dataset.jsonl")


Built 843 instruction examples
First example: {'instruction': 'Classify this news headline into one category: Technology, Markets, Business, Politics, Health, or Energy.', 'input': 'Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors', 'output': 'Business'}
Saved finetune_dataset.jsonl


JSONL instead of one big JSON array because a training script can read it one line at a time instead of loading the entire dataset into memory first. doesn't matter much at 446 rows, but it's the format nearly every real fine-tuning framework expects, so building it this way from the start makes sense.

In [3]:
##-----  chat-template version of the same data -----
##same example as above, reshaped into the messages format expected from chat-tuned models
##same role and content structure used in the TinyLlama chat template
chat_examples = []
for ex in examples:
    chat_examples.append({
        "messages": [
            {"role": "system", "content": "You are a financial news classifier."},
            {"role": "user", "content": f"{ex['instruction']}\n\nHeadline: {ex['input']}"},
            {"role": "assistant", "content": ex["output"],}
        ]
    })

with open("finetune_dataset_chat.jsonl", "w") as f:
    for ex in chat_examples:
        f.write(json.dumps(ex) + "\n")
print("Saved finetune_dataset_chat.jsonl")
print("First chat example:", chat_examples[0])

Saved finetune_dataset_chat.jsonl
First chat example: {'messages': [{'role': 'system', 'content': 'You are a financial news classifier.'}, {'role': 'user', 'content': 'Classify this news headline into one category: Technology, Markets, Business, Politics, Health, or Energy.\n\nHeadline: Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors'}, {'role': 'assistant', 'content': 'Business'}]}


In [4]:
##----- train/validation split -----
##same train/test split concept, applied to fine-tuning data
##the validation set helps catch overfitting during training
from sklearn.model_selection import train_test_split

train_examples, val_examples = train_test_split(examples, test_size=0.1, random_state=42)

with open("finetune_train.jsonl", "w") as f:
    for ex in train_examples:
        f.write(json.dumps(ex) + "\n")

with open("finetune_val.jsonl", "w") as f:
    for ex in val_examples:
        f.write(json.dumps(ex) + "\n")
print(f"Train: {len(train_examples)} examples, Val: {len(val_examples)} examples")

Train: 758 examples, Val: 85 examples


Fine-tuning actually changes the model's weights through real training. Prompting never touches the model at all, it just changes what gets fed into it each time. That's the actual difference between the two.

A full fine-tune retrains every single weight in the model. LoRA gets around that by freezing the whole original model and only training a small set of new adapter layers added on top, way fewer parameters to update, way less memory needed, and the base model itself never changes. QLoRA is LoRA plus quantization, the frozen base model gets loaded at 4-bit precision while the adapters still train at higher precision, cutting memory needs down even further. PEFT is just the name for this whole category of techniques, LoRA and QLoRA both fall under it.

Learning rate, batch size, and epochs still mean the same thing, but the numbers tend to look different in a fine-tuning setup. Learning rates usually end up smaller since the model already knows a lot going in, and epochs tend to be fewer since a smaller dataset can get memorized fast. Adapter saving is one of LoRA's real advantages, only the adapter gets saved, not the whole model, so the saved file ends up way smaller than a full model checkpoint.

Overfitting in a fine-tuning setup means the model starts memorizing the specific training examples instead of learning the actual pattern behind them. A validation set is what catches that, since it's the one part of the data the model never trains on, so checking performance against it during training tells you honestly whether the model's learning something real or just memorizing what it's seen.

## takeaway

closes out with a real, usable dataset instead of a trained model, which matches what the day actually asked for. 446 headlines turned into instruction/response pairs, a chat-formatted version of the same data, and a proper train/validation split, all in JSONL. the workflow write-up covers what would actually happen next: LoRA or QLoRA training against `finetune_train.jsonl`, checked periodically against `finetune_val.jsonl`. next up is RAG, which is a genuinely different approach to the same underlying problem prompting and fine-tuning both run into, a model only knowing what it was trained on.